# 05 - Spark optimization - Solution

Это опорное решение показывает полноценный workflow: baseline -> explain -> optimized code -> config tuning -> benchmark summary.


In [ ]:
import time
from typing import Dict, List

import pandas as pd
from pyspark.sql import SparkSession, DataFrame, functions as F

DATASET_BASE = "/workspace/dataset"
ORDERS_PATH = f"{DATASET_BASE}/orders.csv"
ORDER_ITEMS_PATH = f"{DATASET_BASE}/order_items.csv"
PRODUCTS_PATH = f"{DATASET_BASE}/products.json"
EVENTS_PATH = f"{DATASET_BASE}/events.json"
OUTPUT_BASE = "file:///workspace/output/lesson05"


In [ ]:
BASELINE_CONFIG: Dict[str, str] = {
    "spark.sql.shuffle.partitions": "200",
    "spark.sql.adaptive.enabled": "false",
}

TUNED_CONFIG: Dict[str, str] = {
    "spark.sql.shuffle.partitions": "32",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(50 * 1024 * 1024),
    "spark.sql.files.maxPartitionBytes": str(32 * 1024 * 1024),
}


In [ ]:
def build_spark(app_name: str, config: Dict[str, str]) -> SparkSession:
    builder = SparkSession.builder.appName(app_name)
    for key, value in config.items():
        builder = builder.config(key, value)
    return builder.getOrCreate()


def read_inputs(spark: SparkSession) -> Dict[str, DataFrame]:
    return {
        "orders": spark.read.option("header", True).option("inferSchema", True).csv(ORDERS_PATH),
        "order_items": spark.read.option("header", True).option("inferSchema", True).csv(ORDER_ITEMS_PATH),
        "products": spark.read.json(PRODUCTS_PATH),
        "events": spark.read.json(EVENTS_PATH),
    }


In [ ]:
def build_baseline_pipeline(frames: Dict[str, DataFrame]) -> DataFrame:
    orders_df = frames["orders"]
    order_items_df = frames["order_items"]
    products_df = frames["products"]
    events_df = frames["events"]

    exploded_events = (
        events_df
        .withColumn("item", F.explode_outer("items"))
        .select(
            F.col("event_id"),
            F.col("ts").alias("event_ts"),
            F.col("type").alias("event_type"),
            F.col("user_id").alias("event_user_id"),
            F.col("item.product_id").alias("event_product_id"),
            F.col("item.qty").alias("event_qty"),
            F.col("item.price").alias("event_price"),
            F.col("device"),
        )
    )

    return (
        order_items_df
        .join(orders_df, on="order_id", how="inner")
        .join(products_df, on="product_id", how="left")
        .join(exploded_events, on=order_items_df.product_id == exploded_events.event_product_id, how="left")
        .groupBy("dt", "category", "event_type", "device")
        .agg(
            F.count("*").alias("rows_cnt"),
            F.round(F.sum(F.col("qty") * F.col("price")), 2).alias("revenue"),
            F.round(F.avg("price"), 2).alias("avg_price"),
            F.countDistinct("order_id").alias("orders_cnt"),
            F.countDistinct("user_id").alias("users_cnt"),
        )
        .orderBy("dt", "category", "event_type")
    )


In [ ]:
def build_optimized_pipeline(frames: Dict[str, DataFrame]) -> DataFrame:
    orders_df = frames["orders"].select("order_id", "dt", "user_id")
    order_items_df = frames["order_items"].select("order_id", "product_id", "qty", "price")
    products_df = frames["products"].select("product_id", "category")
    events_df = frames["events"]

    event_items_df = (
        events_df
        .filter(F.col("type").isin("cart", "purchase"))
        .withColumn("item", F.explode_outer("items"))
        .select(
            F.col("type").alias("event_type"),
            F.col("device"),
            F.col("item.product_id").alias("product_id"),
        )
        .dropna(subset=["product_id"])
    )

    fact_df = (
        order_items_df
        .join(orders_df, on="order_id", how="inner")
        .join(F.broadcast(products_df), on="product_id", how="left")
    ).cache()

    return (
        fact_df
        .join(event_items_df.repartition("product_id"), on="product_id", how="left")
        .fillna({"event_type": "no_event", "device": "unknown"})
        .groupBy("dt", "category", "event_type", "device")
        .agg(
            F.count("*").alias("rows_cnt"),
            F.round(F.sum(F.col("qty") * F.col("price")), 2).alias("revenue"),
            F.round(F.avg("price"), 2).alias("avg_price"),
            F.countDistinct("order_id").alias("orders_cnt"),
            F.countDistinct("user_id").alias("users_cnt"),
        )
        .orderBy("dt", "category", "event_type")
    )


In [ ]:
def timed_action(label: str, df: DataFrame) -> Dict[str, float]:
    started = time.perf_counter()
    rows = df.count()
    elapsed = time.perf_counter() - started
    print(f"[{label}] rows={rows} elapsed={elapsed:.2f}s")
    return {"label": label, "rows": rows, "elapsed_sec": round(elapsed, 2)}


def compare_pipeline(config_name: str, config: Dict[str, str]) -> Dict[str, object]:
    spark = build_spark(f"lesson05_{config_name}", config)
    frames = read_inputs(spark)
    baseline_df = build_baseline_pipeline(frames)
    optimized_df = build_optimized_pipeline(frames)

    print(f"=== EXPLAIN baseline / {config_name} ===")
    baseline_df.explain(mode="formatted")
    print(f"=== EXPLAIN optimized / {config_name} ===")
    optimized_df.explain(mode="formatted")

    baseline_metrics = timed_action(f"{config_name}_baseline", baseline_df)
    optimized_metrics = timed_action(f"{config_name}_optimized", optimized_df)

    sample_df = optimized_df.limit(20)
    sample_df.show(truncate=False)

    return {
        "config_name": config_name,
        "shuffle_partitions": config.get("spark.sql.shuffle.partitions"),
        "aqe_enabled": config.get("spark.sql.adaptive.enabled"),
        "baseline_elapsed_sec": baseline_metrics["elapsed_sec"],
        "optimized_elapsed_sec": optimized_metrics["elapsed_sec"],
        "result_rows": optimized_metrics["rows"],
    }


In [ ]:
baseline_report = compare_pipeline("baseline_config", BASELINE_CONFIG)
tuned_report = compare_pipeline("tuned_config", TUNED_CONFIG)

benchmark_df = pd.DataFrame([baseline_report, tuned_report])
benchmark_df


In [ ]:
benchmark_df["improvement_sec"] = (
    benchmark_df["baseline_elapsed_sec"] - benchmark_df["optimized_elapsed_sec"]
)
benchmark_df["improvement_pct_vs_baseline_version"] = (
    benchmark_df["improvement_sec"] / benchmark_df["baseline_elapsed_sec"] * 100
).round(2)
benchmark_df


In [ ]:
engineering_notes = {
    "main_bottlenecks": [
        "Wide joins before projection",
        "Shuffle-heavy aggregation",
        "No broadcast for small dimension",
    ],
    "code_optimizations": [
        "Early select before join",
        "Broadcast small products dimension",
        "Cache reused fact layer",
        "Repartition event-derived layer by product_id",
    ],
    "config_optimizations": [
        "Lower spark.sql.shuffle.partitions",
        "Enable AQE",
        "Enable partition coalescing and skew handling",
        "Tune maxPartitionBytes",
    ],
}
engineering_notes
